## Questão 1


In [ ]:

import numpy as np
import os
import soundfile as sf
from scipy.signal import lfilter

# =====================================================
# 1. FUNÇÕES AUXILIARES
# =====================================================

def apply_delay(x, m):
    """
    Aplica um atraso puro de m amostras ao sinal x
    """
    y = np.zeros(len(x) + m)
    y[m:] = x
    return y


def schroeder_reverb(x, Fs, RT60):
    """
    Reverberador de Schroeder (TP1)
    Implementa o campo reverberante R(z)
    """
    # RT60 = 0 -> espaço livre (sem reverberação)
    if RT60 == 0:
        return np.zeros_like(x)

    # Atrasos típicos do reverberador
    comb_delays = np.array([1557, 1617, 1491, 1422])
    ap_delays   = np.array([225, 556])

    # Ganhos dos filtros comb em função do RT60
    g_comb = 10 ** (-3 * comb_delays / (RT60 * Fs))

    # Banco de filtros comb em paralelo
    y = np.zeros_like(x, dtype=float)
    for d, g in zip(comb_delays, g_comb):
        b = np.zeros(d + 1)
        b[0]  = 1
        b[-1] = -g
        a = np.zeros(d + 1)
        a[0] = 1
        y += lfilter(b, a, x)

    # Filtros all-pass em cascata
    g_ap = 0.7
    for d in ap_delays:
        b = np.zeros(d + 1)
        a = np.zeros(d + 1)
        b[0]  = -g_ap
        b[-1] = 1
        a[0]  = 1
        a[-1] = g_ap
        y = lfilter(b, a, y)

    return y


# =====================================================
# 2. PARÂMETROS FÍSICOS E ACÚSTICOS
# =====================================================

Fs = 48000          # frequência de amostragem [Hz]
c  = 343            # velocidade do som [m/s]

# Distâncias da fonte ao ouvinte (exemplo)
d1 = 1.5            # campo direto
d2 = 2.2            # reflexão 1 (biombo)
d3 = 3.1            # reflexão 2 (biombo)

# Conversão distância -> atraso (amostras)
m1 = int(d1 / c * Fs)
m2 = int(d2 / c * Fs)
m3 = int(d3 / c * Fs)

# Atenuação geométrica
a1 = 1 / d1
a2 = 1 / d2
a3 = 1 / d3


# =====================================================
# 3. LEITURA DO SINAL DE ENTRADA
# =====================================================

# =====================================================
# 3. LEITURA E SOMA DE TODAS AS PISTAS
# =====================================================
folder = "FragileThoughts/"
# Encontra todos os ficheiros .wav na pasta
tracks = [f for f in os.listdir(folder) if f.lower().endswith(".wav")]

if not tracks:
    print("ERRO: Nenhuns ficheiros .wav encontrados na pasta.")
    x = np.zeros(48000)
    Fs = 48000
else:
    print(f"A carregar e somar {len(tracks)} pistas...")
    
    # 1. Ler o primeiro para saber o Fs e o tamanho
    temp_x, Fs = sf.read(os.path.join(folder, tracks[0]))
    if temp_x.ndim > 1: temp_x = temp_x[:, 0] # Forçar Mono
    
    # Acumulador para a soma
    x_soma = np.zeros(len(temp_x))
    
    # 2. Somar todas as pistas
    for t in tracks:
        path = os.path.join(folder, t)
        sig, f = sf.read(path)
        
        # Garantir Mono
        if sig.ndim > 1: sig = sig[:, 0]
        
        # Garantir mesmo tamanho (cortar ou preencher)
        min_len = min(len(x_soma), len(sig))
        x_soma[:min_len] += sig[:min_len]
        print(f" -> Somado: {t}")

    # 3. Normalizar (muito importante porque a soma de 7 pistas distorce)
    peak = np.max(np.abs(x_soma))
    if peak > 0:
        x_soma /= peak
        print(" -> Soma normalizada para evitar clipping.")

    # O nosso sinal de entrada final é a soma de tudo
    x = x_soma

print("Sinal 'x' pronto (Música Completa).")

print("A processar:", tracks[0])


# =====================================================
# QUESTÃO 1 (a)
# CAMPO DIRETO
# =====================================================
# Modelo: a1 * x[n - m1]

campo_direto = a1 * apply_delay(x, m1)
sf.write("Q1_a_campo_direto.wav", campo_direto, Fs)


# =====================================================
# QUESTÃO 1 (b)
# PRIMEIRAS REFLEXÕES (ORDEM 1)
# =====================================================
# Cada reflexão é modelada como um atraso e atenuação
# independentes (modelo FIR)

reflexao_1 = a2 * apply_delay(x, m2)
reflexao_2 = a3 * apply_delay(x, m3)

# Igualar comprimentos
L = max(len(reflexao_1), len(reflexao_2))
reflexao_1 = np.pad(reflexao_1, (0, L - len(reflexao_1)))
reflexao_2 = np.pad(reflexao_2, (0, L - len(reflexao_2)))

campo_reflexoes = reflexao_1 + reflexao_2
sf.write("Q1_b_primeiras_reflexoes.wav", campo_reflexoes, Fs)


# =====================================================
# QUESTÃO 1 (c) - CORRIGIDA (NORMALIZADA)
# CAMPO REVERBERANTE (R(z))
# =====================================================
print("A gerar Reverbs isolados (Q1c)...")

RT60_values = [0, 0.5, 2, 10]

for RT in RT60_values:
    # 1. Gerar Reverb
    campo_reverberante = schroeder_reverb(x, Fs, RT)
    
    # 2. NORMALIZAÇÃO INDIVIDUAL
    # Antes de gravar, verificamos o volume máximo.
    # Se for muito alto, baixamos tudo proporcionalmente.
    peak = np.max(np.abs(campo_reverberante))
    
    if peak > 0: # Evita erro de divisão por zero no caso RT=0
        # Normalizamos para 0.9 (90% do volume) para não ficar no limite
        campo_reverberante = (campo_reverberante / peak) * 0.9
        print(f" -> Q1c RT{RT}: Normalizado (Pico era {peak:.2f})")
    
    sf.write(f"Q1_c_campo_reverberante_RT{RT}.wav",
             campo_reverberante, Fs)


# =====================================================
# QUESTÃO 1 (d) - CORRIGIDA (SEM ESTOURAR)
# CAMPO TOTAL
# =====================================================

print("A gerar Campo Total (com proteção de volume)...")

for RT in RT60_values:
    # 1. Gerar o Reverb
    campo_reverberante = schroeder_reverb(x, Fs, RT)

    # 2. Igualar comprimentos (Padding)
    L = max(len(campo_direto), 
            len(campo_reflexoes), 
            len(campo_reverberante))

    cd = np.pad(campo_direto, (0, L - len(campo_direto)))
    cr = np.pad(campo_reflexoes, (0, L - len(campo_reflexoes)))
    rv = np.pad(campo_reverberante, (0, L - len(campo_reverberante)))

    # 3. MISTURA CONTROLADA (WET/DRY)
    # O Reverb (Wet) deve ser mais baixo que o som direto (Dry).
    # Multiplicamos o reverb por 0.05 (5% do volume) ou 0.1.
    wet_gain = 0.05 
    
    if RT == 0: 
        wet_gain = 0 # Se RT é 0, não há reverb nenhum
    
    # Soma ponderada
    campo_total = cd + cr + (rv * wet_gain)

    # 4. NORMALIZAÇÃO DE SEGURANÇA
    # Verifica qual é o pico máximo do sinal
    peak = np.max(np.abs(campo_total))
    
    # Se passar de 1.0, divide tudo pelo pico para trazer o máximo para 1.0
    if peak > 1.0:
        campo_total /= peak
        print(f" -> RT={RT}s: Sinal normalizado (Pico era {peak:.2f})")

    sf.write(f"Q1_d_campo_total_RT{RT}.wav", campo_total, Fs)

print("Questão 1 (d) concluída sem distorção.")


## Questão 2

In [ ]:
import numpy as np
import soundfile as sf
import os
from scipy.signal import lfilter

# =====================================================
# 1. CONFIGURAÇÃO
# =====================================================
# Caminho para a pasta com os ficheiros .dat (ex: H0e090a.dat)
# Certifique-se que aponta para a pasta 'elev0'
PASTA_HRTF = "HRTF/compact/elev0/" 

# =====================================================
# 2. FUNÇÕES AUXILIARES
# =====================================================

def apply_delay(x, m):
    """Aplica atraso de m amostras"""
    y = np.zeros(len(x) + m)
    y[m:] = x
    return y

def schroeder_reverb(x, Fs, RT60):
    """Gera a cauda de reverberação (Mono)"""
    if RT60 == 0: return np.zeros_like(x)

    comb_delays = np.array([1557, 1617, 1491, 1422])
    ap_delays   = np.array([225, 556])
    g_comb = 10 ** (-3 * comb_delays / (RT60 * Fs))

    y = np.zeros_like(x, dtype=float)
    for d, g in zip(comb_delays, g_comb):
        b, a = np.zeros(d + 1), np.zeros(d + 1)
        b[0], b[-1] = 1, -g
        a[0] = 1
        y += lfilter(b, a, x)

    g_ap = 0.7
    for d in ap_delays:
        b, a = np.zeros(d + 1), np.zeros(d + 1)
        b[0], b[-1] = -g_ap, 1
        a[0], a[-1] = 1, g_ap
        y = lfilter(b, a, y)
    return y

def binaural_hrtf_dat(x, angle_deg):
    """
    Lê HRTF .dat com SIMETRIA para bases de dados 0-180º.
    """
    # 1. Normalizar ângulo para 0-359
    # O passo é 5 graus (round/5)
    angle_clean = int(5 * round(angle_deg / 5)) % 360
    
    # 2. Lógica de Simetria (Crucial para a sua base de dados!)
    # Se o ângulo for > 180 (lado direito), usamos o equivalente do lado esquerdo
    # e trocamos os canais.
    # Ex: 300º (Direita) -> Usamos 60º (Esquerda) e trocamos L com R.
    swap_channels = False
    
    if angle_clean > 180:
        angle_clean = 360 - angle_clean
        swap_channels = True
        
    # Nome do ficheiro (agora nunca passará de 180)
    filename = f"H0e{angle_clean:03d}a.dat"
    filepath = os.path.join(PASTA_HRTF, filename)

    # 3. Leitura do ficheiro RAW
    if not os.path.exists(filepath):
        print(f"AVISO CRÍTICO: {filename} não existe.Ângulo original: {angle_deg}º.")
        return x, x # Fallback Mono
        
    try:
        # Ler Raw Big-Endian 16-bit
        raw_data = np.fromfile(filepath, dtype='>i2')
        hrtf = raw_data.astype(float) / 32768.0
        hrtf = hrtf.reshape(-1, 2)
        
        # Separar canais da HRTF
        ir_L = hrtf[:, 0]
        ir_R = hrtf[:, 1]
        
        # 4. Trocar canais se estivemos a usar simetria
        if swap_channels:
            ir_L, ir_R = ir_R, ir_L
            
        # 5. Convolução
        xL = np.convolve(x, ir_L, mode='full')
        xR = np.convolve(x, ir_R, mode='full')
        
        return xL, xR
        
    except Exception as e:
        print(f"Erro ao ler {filename}: {e}")
        return x, x

# =====================================================
# 3. EXECUÇÃO
# =====================================================
# Carregar som (tenta ler FragileThoughts ou cria dummy)
folder = "FragileThoughts/"
files = [f for f in os.listdir(folder) if f.endswith(".wav")]

if files:
    print(f"A processar {len(files)} ficheiros...")

    # 1. Ler o primeiro ficheiro para definir o tamanho e o Fs
    first_path = os.path.join(folder, files[0])
    x_soma, Fs = sf.read(first_path)
    
    # Garantir que é mono
    if x_soma.ndim > 1: 
        x_soma = x_soma[:, 0]
    
    # 2. Iterar sobre os restantes ficheiros (do índice 1 até ao fim)
    for filename in files[1:]:
        path = os.path.join(folder, filename)
        sig, _ = sf.read(path) # O Fs assume-se igual para todos
        
        # Garantir mono
        if sig.ndim > 1: 
            sig = sig[:, 0]
            
        # 3. Somar com segurança de tamanho
        # (Caso as pistas tenham tamanhos ligeiramente diferentes)
        min_len = min(len(x_soma), len(sig))
        x_soma = x_soma[:min_len] # Corta o acumulador se for maior
        x_soma += sig[:min_len]   # Soma apenas até onde ambos existem
        
        print(f" -> Somado: {filename}")

    # 4. Atribuir à variável final 'x'
    x = x_soma

    # 5. IMPORTANTE: Normalizar
    # Somar 7 pistas vai fazer o volume explodir (clipping). 
    # Temos de trazer o pico máximo para 1.0 ou menos.
    peak = np.max(np.abs(x))
    if peak > 1.0:
        x = x / peak
        print(" -> Sinal normalizado (para evitar distorção).")

else:
    print("A usar sinal de teste (sem ficheiros na pasta).")
    Fs = 48000
    x = np.zeros(Fs)

# Parâmetros Q1
c = 343
d1, d2, d3 = 1.5, 2.2, 3.1
m1, m2, m3 = int(d1/c*Fs), int(d2/c*Fs), int(d3/c*Fs)
ang_direto, ang_ref1, ang_ref2 = 30, -60, 110 # -60 vai virar 300, e depois 60 (simétrico)

# --- ALÍNEA A (Absorção) ---
print("A processar Q2a...")
alphas = [0, 0.5, 0.8, 1]
base_binaural = None

for alpha in alphas:
    # Ganhos
    g1 = 1.0/d1
    g2 = (1.0-alpha)/d2
    g3 = (1.0-alpha)/d3
    
    # Sinais
    s_d  = g1 * apply_delay(x, m1)
    s_r1 = g2 * apply_delay(x, m2)
    s_r2 = g3 * apply_delay(x, m3)
    
    # Espacialização (Agora com simetria automática!)
    Ld, Rd   = binaural_hrtf_dat(s_d, ang_direto)
    Lr1, Rr1 = binaural_hrtf_dat(s_r1, ang_ref1)
    Lr2, Rr2 = binaural_hrtf_dat(s_r2, ang_ref2)
    
    # Soma
    maxlen = max(len(Ld), len(Lr1), len(Lr2))
    def p(s): return np.pad(s, (0, maxlen - len(s)))
    
    mix_L = p(Ld) + p(Lr1) + p(Lr2)
    mix_R = p(Rd) + p(Rr1) + p(Rr2)
    
    # Gravar
    out = np.column_stack((mix_L, mix_R))
    if np.max(np.abs(out)) > 0: out /= np.max(np.abs(out))
    sf.write(f"Q2a_Final_Alpha{alpha}.wav", out, Fs)
    
    if alpha == 0.5: base_binaural = (mix_L, mix_R)

# --- ALÍNEA B (Reverb) ---
print("A processar Q2b...")
RT60_vals = [0, 0.5, 2, 10]
bL, bR = base_binaural if base_binaural else (np.zeros(100), np.zeros(100))

for RT in RT60_vals:
    if RT == 0:
        fL, fR = bL, bR
    else:
        rev_mono = schroeder_reverb(x, Fs, RT)
        # Reverb a 0 graus (sem simetria necessária, pois 0 < 180)
        rL, rR = binaural_hrtf_dat(rev_mono, 0)
        
        wet = 0.05
        mtot = max(len(bL), len(rL))
        def p(s): return np.pad(s, (0, mtot - len(s)))
        
        fL = p(bL) + p(rL)*wet
        fR = p(bR) + p(rR)*wet
        
    out = np.column_stack((fL, fR))
    if np.max(np.abs(out)) > 0: out /= np.max(np.abs(out))
    sf.write(f"Q2b_Final_RT{RT}.wav", out, Fs)

print("Questão 2 Concluída (Simetria aplicada).")

## Questão 3

In [ ]:
import numpy as np
import soundfile as sf

# =====================================================
# QUESTÃO 3
# Reprodução Ambisonic em sistema multicanal (8.0)
# =====================================================

print("A iniciar processamento Ambisonic...")

# 1. RECUPERAR SINAIS E DADOS DAS QUESTÕES ANTERIORES
# -----------------------------------------------------
# (Assume-se que as variáveis campo_direto, reflexao_1, reflexao_2,
#  e os ângulos ang_direto, ang_ref1, ang_ref2 já existem na memória
#  após correr a Q1 e Q2.).

# Assegurar que os ângulos estão em radianos
theta_d  = np.deg2rad(ang_direto) # 30º
theta_r1 = np.deg2rad(ang_ref1)   # -60º
theta_r2 = np.deg2rad(ang_ref2)   # 110º

# Igualar o comprimento dos vetores (Padding) para poder somar
L_max = max(len(campo_direto), len(reflexao_1), len(reflexao_2))

s_d  = np.pad(campo_direto, (0, L_max - len(campo_direto)))
s_r1 = np.pad(reflexao_1,   (0, L_max - len(reflexao_1)))
s_r2 = np.pad(reflexao_2,   (0, L_max - len(reflexao_2)))


# 2. CODIFICAÇÃO (B-FORMAT - 1ª ORDEM)
# -----------------------------------------------------
# W = S / sqrt(2)
# X = S * cos(theta)
# Y = S * sin(theta)

# Calculamos W, X, Y totais (soma das 3 fontes sonoras codificadas)
W = (s_d + s_r1 + s_r2) / np.sqrt(2)

X = (s_d * np.cos(theta_d)) + \
    (s_r1 * np.cos(theta_r1)) + \
    (s_r2 * np.cos(theta_r2))

Y = (s_d * np.sin(theta_d)) + \
    (s_r1 * np.sin(theta_r1)) + \
    (s_r2 * np.sin(theta_r2))

# Guardar o B-Format (opcional, para análise)
sf.write("Q3_Ambisonic_BFormat_WXY.wav", 
         np.column_stack((W, X, Y)), Fs)


# 3. GEOMETRIA DOS ALTIFALANTES
# -----------------------------------------------------
# 8 altifalantes em círculo, igualmente espaçados (45º entre cada)
# 0º geralmente é Frente. Sentido anti-horário.
num_speakers = 8
spk_angles_deg = np.linspace(0, 360, num_speakers, endpoint=False)
spk_angles_rad = np.deg2rad(spk_angles_deg)


# 4. DESCODIFICAÇÃO (VIRTUAL MICROPHONES)
# -----------------------------------------------------
# Slide 40: Microfone virtual com p=0.5 (Cardioide)
# Sinal = p * Omni + (1-p) * Figure8_na_direcao_do_altifalante
p = 0.5 

speaker_signals = []

for phi in spk_angles_rad:
    # Componente Omnidirecional (recuperada de W)
    omni_comp = np.sqrt(2) * W
    
    # Componente Figura-8 projetada na direção do altifalante (phi)
    fig8_comp = X * np.cos(phi) + Y * np.sin(phi)
    
    # Soma ponderada pelo fator de diretividade p
    sig = p * omni_comp + (1 - p) * fig8_comp
    
    speaker_signals.append(sig)

# Converter lista para matriz (amostras x 8 canais)
multichannel_out = np.column_stack(speaker_signals)

# Normalização de segurança (para evitar clipping no wav multicanal)
peak = np.max(np.abs(multichannel_out))
if peak > 1.0:
    multichannel_out /= peak
    print(f"Sinal normalizado (Peak attenuation: {1/peak:.2f})")

# 5. ESCRITA DO FICHEIRO FINAL
# -----------------------------------------------------
output_filename = "Q3_Ambisonic_Decoded_8ch.wav"
sf.write(output_filename, multichannel_out, Fs)

print(f"Questão 3 concluída.")
print(f"Ficheiro gerado: {output_filename} ({num_speakers} canais).")
print(f"Ficheiro B-Format gerado: Q3_Ambisonic_BFormat_WXY.wav")